# Validation, bootstrap & leakage: practice notebook

4 exercises, each with hidden tests. Data is loaded as **polars**; pandas only appears where a library insists on it.

**How to use it**
1. Run the three setup cells below once (Runtime → Run all works too; the exercise cells just print `Wrong answer` until you solve them).
2. For each exercise, write your code in the solution cell, run it, then run the `check(...)` cell under it. It prints **Accepted** or which test failed and why.
3. Stuck? `solution("w1")` prints the reference solution.

Opened from the roadmap it runs in Google Colab. To keep your work, use File → Save a copy in Drive the first time.

In [ ]:
# Install anything missing (Colab already has numpy, scipy, statsmodels, scikit-learn, matplotlib)
import importlib.util, subprocess, sys
for _p in ['polars']:
    if importlib.util.find_spec(_p) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p], check=False)


## Setup: data
The data every exercise uses. Read it if you like; don't change it.

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import statsmodels.api as sm

def rdata(name, package):
    """Load an R dataset (via statsmodels / Rdatasets) as a polars DataFrame, NaN → null."""
    df = pl.from_pandas(sm.datasets.get_rdataset(name, package, cache=True).data.reset_index(drop=True)).drop("rownames", strict=False)
    return df.with_columns(pl.col(pl.Float64).fill_nan(None))

default = rdata("Default", "ISLR")
_g = np.random.default_rng(21)
noise_X = _g.normal(size=(100, 5000))
noise_y = _g.integers(0, 2, 100)
walk = np.cumsum(_g.normal(size=600))

In [ ]:
# Test runner: check(id) and solution(id)
import base64 as _b64, json as _json, io as _io, contextlib as _ctx, traceback as _tb
_LAB = _json.loads(_b64.b64decode("eyJ3MSI6IHsic29sdXRpb24iOiAiZGVmIGluX2Jvb3RfcHJvYihuKTpcbiAgICByZXR1cm4gMSAtICgxIC0gMSAvIG4pICoqIG5cblxuZGVmIGluX2Jvb3Rfc2ltKG49MTAwLCBqPTQsIHJlcHM9MTBfMDAwLCBzZWVkPTApOlxuICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKVxuICAgIHJldHVybiBmbG9hdChucC5tZWFuKFsocm5nLmludGVnZXJzKDEsIG4gKyAxLCBuKSA9PSBqKS5hbnkoKSBmb3IgXyBpbiByYW5nZShyZXBzKV0pKVxuIiwgInRlc3RzIjogIlxuZGVmIGNsb3NlKGEsIGIsIHJlbD0xZS02LCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhID0gZmxvYXQoYSk7IGIgPSBmbG9hdChiKVxuICAgIGFzc2VydCBhYnMoYSAtIGIpIDw9IHJlbCAqIG1heCgxLjAsIGFicyhiKSksIGZcInt3aGF0fSA9IHthOi42Z30sIGV4cGVjdGVkIHtiOi42Z31cIlxuZGVmIGJldHdlZW4oYSwgbG8sIGhpLCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgbG8gPD0gZmxvYXQoYSkgPD0gaGksIGZcInt3aGF0fSA9IHtmbG9hdChhKTouNGZ9LCBleHBlY3RlZCBiZXR3ZWVuIHtsb30gYW5kIHtoaX1cIlxuZGVmIHNhbWVfa2V5cyhhLCBiLCB3aGF0PVwicmVzdWx0XCIpOlxuICAgIGFzc2VydCBpc2luc3RhbmNlKGEsIGRpY3QpLCBmXCJ7d2hhdH0gc2hvdWxkIGJlIGEgZGljdCwgZ290IHt0eXBlKGEpLl9fbmFtZV9ffVwiXG4gICAgYXNzZXJ0IHNldChhKSA9PSBzZXQoYiksIGZcInt3aGF0fSBrZXlzIHtzb3J0ZWQoYSl9LCBleHBlY3RlZCB7c29ydGVkKGIpfVwiXG5cbmRlZiBfdCgpOlxuICAgIGZvciBuIGluICgxLCA1LCAxMDAsIDEwXzAwMCk6IGNsb3NlKFVbXCJpbl9ib290X3Byb2JcIl0obiksIFJbXCJpbl9ib290X3Byb2JcIl0obiksIDFlLTksIGZcImluX2Jvb3RfcHJvYih7bn0pXCIpXG4gICAgYmV0d2VlbihVW1wiaW5fYm9vdF9zaW1cIl0oc2VlZD0yKSwgMC42MiwgMC42NSwgXCJzaW11bGF0ZWQgcHJvYmFiaWxpdHkgZm9yIG4gPSAxMDBcIilcblRFU1RTID0gWyhcImZvcm11bGEgYW5kIHNpbXVsYXRpb25cIiwgX3QpXVxuIiwgImxhbmciOiAicHkifSwgIncyIjogeyJzb2x1dGlvbiI6ICJkZWYgX3h5KGRmKTpcbiAgICBYID0gc20uYWRkX2NvbnN0YW50KGRmLnNlbGVjdChcImluY29tZVwiLCBcImJhbGFuY2VcIikudG9fbnVtcHkoKSlcbiAgICB5ID0gKGRmW1wiZGVmYXVsdFwiXSA9PSBcIlllc1wiKS5jYXN0KHBsLkludDY0KS50b19udW1weSgpXG4gICAgcmV0dXJuIFgsIHlcblxuZGVmIGxvZ2l0X3NlKGRmKTpcbiAgICBYLCB5ID0gX3h5KGRmKVxuICAgIHNlID0gc20uTG9naXQoeSwgWCkuZml0KGRpc3A9MCkuYnNlXG4gICAgcmV0dXJuIGZsb2F0KHNlWzFdKSwgZmxvYXQoc2VbMl0pXG5cbmRlZiBib290X3NlKGRmLCBCPTEwMCwgc2VlZD0wKTpcbiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZClcbiAgICBuID0gZGYuaGVpZ2h0XG4gICAgY29lZnMgPSBbXVxuICAgIGZvciBfIGluIHJhbmdlKEIpOlxuICAgICAgICBYLCB5ID0gX3h5KGRmW3JuZy5pbnRlZ2VycygwLCBuLCBuKV0pXG4gICAgICAgIGNvZWZzLmFwcGVuZChzbS5Mb2dpdCh5LCBYKS5maXQoZGlzcD0wKS5wYXJhbXNbMTpdKVxuICAgIHMgPSBucC5zdGQoY29lZnMsIGF4aXM9MCwgZGRvZj0xKVxuICAgIHJldHVybiBmbG9hdChzWzBdKSwgZmxvYXQoc1sxXSlcbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX3QoKTpcbiAgICBhLCBiID0gVVtcImxvZ2l0X3NlXCJdKGRlZmF1bHQpLCBSW1wibG9naXRfc2VcIl0oZGVmYXVsdClcbiAgICBjbG9zZShhWzBdLCBiWzBdLCAxZS02LCBcImluY29tZSBTRVwiKTsgY2xvc2UoYVsxXSwgYlsxXSwgMWUtNiwgXCJiYWxhbmNlIFNFXCIpXG4gICAgcyA9IFVbXCJib290X3NlXCJdKGRlZmF1bHQsIEI9NjAsIHNlZWQ9NClcbiAgICBjbG9zZShzWzBdLCBiWzBdLCAwLjM1LCBcImJvb3RzdHJhcCBpbmNvbWUgU0VcIik7IGNsb3NlKHNbMV0sIGJbMV0sIDAuMzUsIFwiYm9vdHN0cmFwIGJhbGFuY2UgU0VcIilcblRFU1RTID0gWyhcImZvcm11bGEgdnMgYm9vdHN0cmFwXCIsIF90KV1cbiIsICJsYW5nIjogInB5In0sICJ3MyI6IHsic29sdXRpb24iOiAiZnJvbSBza2xlYXJuLmZlYXR1cmVfc2VsZWN0aW9uIGltcG9ydCBTZWxlY3RLQmVzdCwgZl9jbGFzc2lmXG5mcm9tIHNrbGVhcm4ubGluZWFyX21vZGVsIGltcG9ydCBMb2dpc3RpY1JlZ3Jlc3Npb25cbmZyb20gc2tsZWFybi5waXBlbGluZSBpbXBvcnQgUGlwZWxpbmVcbmZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IFN0cmF0aWZpZWRLRm9sZCwgY3Jvc3NfdmFsX3Njb3JlXG5cbkNWID0gU3RyYXRpZmllZEtGb2xkKDUsIHNodWZmbGU9VHJ1ZSwgcmFuZG9tX3N0YXRlPTApXG5cbmRlZiBsZWFreV9jdihYLCB5KTpcbiAgICBYcyA9IFNlbGVjdEtCZXN0KGZfY2xhc3NpZiwgaz0yMCkuZml0X3RyYW5zZm9ybShYLCB5KVxuICAgIHJldHVybiBmbG9hdChjcm9zc192YWxfc2NvcmUoTG9naXN0aWNSZWdyZXNzaW9uKCksIFhzLCB5LCBjdj1DVikubWVhbigpKVxuXG5kZWYgaG9uZXN0X2N2KFgsIHkpOlxuICAgIHBpcGUgPSBQaXBlbGluZShbKFwic2VsZWN0XCIsIFNlbGVjdEtCZXN0KGZfY2xhc3NpZiwgaz0yMCkpLCAoXCJsclwiLCBMb2dpc3RpY1JlZ3Jlc3Npb24oKSldKVxuICAgIHJldHVybiBmbG9hdChjcm9zc192YWxfc2NvcmUocGlwZSwgWCwgeSwgY3Y9Q1YpLm1lYW4oKSlcbiIsICJ0ZXN0cyI6ICJcbmRlZiBjbG9zZShhLCBiLCByZWw9MWUtNiwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYSA9IGZsb2F0KGEpOyBiID0gZmxvYXQoYilcbiAgICBhc3NlcnQgYWJzKGEgLSBiKSA8PSByZWwgKiBtYXgoMS4wLCBhYnMoYikpLCBmXCJ7d2hhdH0gPSB7YTouNmd9LCBleHBlY3RlZCB7YjouNmd9XCJcbmRlZiBiZXR3ZWVuKGEsIGxvLCBoaSwgd2hhdD1cInZhbHVlXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInlvdXIgZnVuY3Rpb24gcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGxvIDw9IGZsb2F0KGEpIDw9IGhpLCBmXCJ7d2hhdH0gPSB7ZmxvYXQoYSk6LjRmfSwgZXhwZWN0ZWQgYmV0d2VlbiB7bG99IGFuZCB7aGl9XCJcbmRlZiBzYW1lX2tleXMoYSwgYiwgd2hhdD1cInJlc3VsdFwiKTpcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBkaWN0KSwgZlwie3doYXR9IHNob3VsZCBiZSBhIGRpY3QsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBzZXQoYSkgPT0gc2V0KGIpLCBmXCJ7d2hhdH0ga2V5cyB7c29ydGVkKGEpfSwgZXhwZWN0ZWQge3NvcnRlZChiKX1cIlxuXG5kZWYgX3QoKTpcbiAgICBiZXR3ZWVuKFVbXCJsZWFreV9jdlwiXShub2lzZV9YLCBub2lzZV95KSwgMC43NSwgMS4wLCBcImxlYWt5IENWIGFjY3VyYWN5XCIpXG4gICAgYmV0d2VlbihVW1wiaG9uZXN0X2N2XCJdKG5vaXNlX1gsIG5vaXNlX3kpLCAwLjI1LCAwLjcsIFwiaG9uZXN0IENWIGFjY3VyYWN5XCIpXG5URVNUUyA9IFsoXCJsZWFreSBsb29rcyBncmVhdCwgaG9uZXN0IGlzIGEgY29pbiBmbGlwXCIsIF90KV1cbiIsICJsYW5nIjogInB5In0sICJ3NCI6IHsic29sdXRpb24iOiAiZnJvbSBza2xlYXJuLmVuc2VtYmxlIGltcG9ydCBSYW5kb21Gb3Jlc3RSZWdyZXNzb3JcbmZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IEtGb2xkLCBUaW1lU2VyaWVzU3BsaXQsIGNyb3NzX3ZhbF9zY29yZVxuXG5kZWYgY3ZfcjIoc2VyaWVzKTpcbiAgICBYID0gbnAuYXJhbmdlKGxlbihzZXJpZXMpKS5yZXNoYXBlKC0xLCAxKVxuICAgIG1vZGVsID0gUmFuZG9tRm9yZXN0UmVncmVzc29yKG5fZXN0aW1hdG9ycz0xMDAsIHJhbmRvbV9zdGF0ZT0wKVxuICAgIHNodWZmbGVkID0gY3Jvc3NfdmFsX3Njb3JlKG1vZGVsLCBYLCBzZXJpZXMsIGN2PUtGb2xkKDUsIHNodWZmbGU9VHJ1ZSwgcmFuZG9tX3N0YXRlPTApLCBzY29yaW5nPVwicjJcIikubWVhbigpXG4gICAgZm9yd2FyZCA9IGNyb3NzX3ZhbF9zY29yZShtb2RlbCwgWCwgc2VyaWVzLCBjdj1UaW1lU2VyaWVzU3BsaXQoNSksIHNjb3Jpbmc9XCJyMlwiKS5tZWFuKClcbiAgICByZXR1cm4gZmxvYXQoc2h1ZmZsZWQpLCBmbG9hdChmb3J3YXJkKVxuIiwgInRlc3RzIjogIlxuZGVmIGNsb3NlKGEsIGIsIHJlbD0xZS02LCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhID0gZmxvYXQoYSk7IGIgPSBmbG9hdChiKVxuICAgIGFzc2VydCBhYnMoYSAtIGIpIDw9IHJlbCAqIG1heCgxLjAsIGFicyhiKSksIGZcInt3aGF0fSA9IHthOi42Z30sIGV4cGVjdGVkIHtiOi42Z31cIlxuZGVmIGJldHdlZW4oYSwgbG8sIGhpLCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgbG8gPD0gZmxvYXQoYSkgPD0gaGksIGZcInt3aGF0fSA9IHtmbG9hdChhKTouNGZ9LCBleHBlY3RlZCBiZXR3ZWVuIHtsb30gYW5kIHtoaX1cIlxuZGVmIHNhbWVfa2V5cyhhLCBiLCB3aGF0PVwicmVzdWx0XCIpOlxuICAgIGFzc2VydCBpc2luc3RhbmNlKGEsIGRpY3QpLCBmXCJ7d2hhdH0gc2hvdWxkIGJlIGEgZGljdCwgZ290IHt0eXBlKGEpLl9fbmFtZV9ffVwiXG4gICAgYXNzZXJ0IHNldChhKSA9PSBzZXQoYiksIGZcInt3aGF0fSBrZXlzIHtzb3J0ZWQoYSl9LCBleHBlY3RlZCB7c29ydGVkKGIpfVwiXG5cbmRlZiBfdCgpOlxuICAgIGEsIGIgPSBVW1wiY3ZfcjJcIl0od2FsaylcbiAgICBiZXR3ZWVuKGEsIDAuOSwgMS4wLCBcInNodWZmbGVkIEstZm9sZCBSXHUwMGIyXCIpXG4gICAgYXNzZXJ0IGIgPCAwLjMsIGZcIndhbGstZm9yd2FyZCBSXHUwMGIyID0ge2I6LjNmfSwgZXhwZWN0ZWQgd2VsbCBiZWxvdyB0aGUgc2h1ZmZsZWQgb25lXCJcbiAgICByID0gUltcImN2X3IyXCJdKHdhbGspXG4gICAgY2xvc2UoYSwgclswXSwgMWUtNiwgXCJzaHVmZmxlZCBSXHUwMGIyXCIpOyBjbG9zZShiLCByWzFdLCAxZS02LCBcIndhbGstZm9yd2FyZCBSXHUwMGIyXCIpXG5URVNUUyA9IFsoXCJzaHVmZmxlZCBsZWFrcywgd2Fsay1mb3J3YXJkIGRvZXNuJ3RcIiwgX3QpXVxuIiwgImxhbmciOiAicHkifX0=").decode())
_SETUP = _b64.b64decode("aW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwb2xhcnMgYXMgcGwKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdAppbXBvcnQgc3RhdHNtb2RlbHMuYXBpIGFzIHNtCgpkZWYgcmRhdGEobmFtZSwgcGFja2FnZSk6CiAgICAiIiJMb2FkIGFuIFIgZGF0YXNldCAodmlhIHN0YXRzbW9kZWxzIC8gUmRhdGFzZXRzKSBhcyBhIHBvbGFycyBEYXRhRnJhbWUsIE5hTiDihpIgbnVsbC4iIiIKICAgIGRmID0gcGwuZnJvbV9wYW5kYXMoc20uZGF0YXNldHMuZ2V0X3JkYXRhc2V0KG5hbWUsIHBhY2thZ2UsIGNhY2hlPVRydWUpLmRhdGEucmVzZXRfaW5kZXgoZHJvcD1UcnVlKSkuZHJvcCgicm93bmFtZXMiLCBzdHJpY3Q9RmFsc2UpCiAgICByZXR1cm4gZGYud2l0aF9jb2x1bW5zKHBsLmNvbChwbC5GbG9hdDY0KS5maWxsX25hbihOb25lKSkKCmRlZmF1bHQgPSByZGF0YSgiRGVmYXVsdCIsICJJU0xSIikKX2cgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoMjEpCm5vaXNlX1ggPSBfZy5ub3JtYWwoc2l6ZT0oMTAwLCA1MDAwKSkKbm9pc2VfeSA9IF9nLmludGVnZXJzKDAsIDIsIDEwMCkKd2FsayA9IG5wLmN1bXN1bShfZy5ub3JtYWwoc2l6ZT02MDApKQ==").decode()

def _err(e):
    fr = [f for f in _tb.extract_tb(e.__traceback__) if f.filename.startswith("<ipython") or f.filename == "<string>"]
    return f"{type(e).__name__}: {e}"

def check(ex_id, query=None):
    """Run the hidden tests for one exercise, like Submit in the editor."""
    e = _LAB[ex_id]
    ref = {}
    with _ctx.redirect_stdout(_io.StringIO()):
        exec(_SETUP, ref)
        if e["lang"] == "sql":
            ref["__sql__"] = e["solution"]
        else:
            exec(e["solution"], ref)
    if e["lang"] == "sql":
        if query is None:
            print("Pass your query string: check(\"" + ex_id + "\", q)")
            return
        user = {"__sql__": query}
    else:
        user = dict(globals())
    tns = dict(ref)
    tns["U"], tns["R"] = user, ref
    exec(e["tests"], tns)
    ok, tests = 0, tns["TESTS"]
    for name, fn in tests:
        try:
            fn()
            print("  ✓ " + name)
            ok += 1
        except AssertionError as err:
            print("  ✗ " + name + ": " + (str(err) or "wrong answer"))
        except Exception as err:
            print("  ✗ " + name + ": " + _err(err))
    print(("Accepted" if ok == len(tests) else "Wrong answer") + f" · {ok} / {len(tests)} test cases passed")

def solution(ex_id):
    """Print the reference solution. Try once more first."""
    print(_LAB[ex_id]["solution"])

---
## 1. ISLP 5 ex. 2: probability an observation is in a bootstrap sample
*Easy · id `w1`*

<p>ISLP chapter 5, conceptual exercise 2 (parts a–h). Do a–f on paper first.</p>
<p>Write <code>in_boot_prob(n)</code> = 1 − (1 − 1/n)ⁿ, and <code>in_boot_sim(n=100, j=4, reps=10_000, seed=0)</code>: the fraction of <code>reps</code> bootstrap samples (<code>rng.integers(1, n + 1, n)</code>) that contain observation j. Plot <code>in_boot_prob</code> for n = 1..100,000. What is the limit?</p>

**Theory to know cold**
<ul><li>(1 − 1/n)ⁿ → e⁻¹, so about 63.2% of observations appear in each bootstrap sample and 36.8% are out-of-bag.</li></ul>

In [ ]:
def in_boot_prob(n):
    # your code here
    pass

def in_boot_sim(n=100, j=4, reps=10_000, seed=0):
    rng = np.random.default_rng(seed)
    # your code here
    pass

# Sample run
try:
    print(in_boot_prob(5), in_boot_prob(100), in_boot_prob(10_000), in_boot_sim())
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("w1")

---
## 2. ISLP 5 ex. 6: bootstrap standard errors for logistic regression on Default
*Medium · id `w2`*

<p>ISLP chapter 5, applied exercise 6. Model P(default = Yes) from <code>income</code> and <code>balance</code>.</p>
<p>Write <code>logit_se(df)</code>: fit <code>sm.Logit</code> with a constant and return the formula standard errors of income and balance as a tuple. Write <code>boot_se(df, B=100, seed=0)</code>: refit on B bootstrap resamples (row indices <code>rng.integers(0, n, n)</code>, take rows with polars <code>df[idx]</code>) and return the standard deviations of the two coefficients.</p>
<p>Do the two agree? When would they not?</p>

**Theory to know cold**
<ul><li>The bootstrap SE needs no formula, only the ability to refit; it is right even when model assumptions are off, as long as rows are independent.</li><li>For time series, resample blocks, not rows.</li></ul>

In [ ]:
def _xy(df):
    X = sm.add_constant(df.select("income", "balance").to_numpy())
    y = (df["default"] == "Yes").cast(pl.Int64).to_numpy()
    return X, y

def logit_se(df: pl.DataFrame):
    # your code here
    pass

def boot_se(df: pl.DataFrame, B=100, seed=0):
    rng = np.random.default_rng(seed)
    # your code here
    pass

# Sample run
try:
    print(logit_se(default)); print(boot_se(default))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("w2")

---
## 3. Leakage: selecting features before cross-validation
*Medium · id `w3`*

<p><code>noise_X</code> has 5,000 pure-noise features for 100 samples; <code>noise_y</code> is a random 0/1 label. No model can beat 50%.</p>
<p>1. <code>leaky_cv(X, y)</code>: pick the 20 features with the highest <code>f_classif</code> score <b>on all the data</b>, then return the mean 5-fold CV accuracy of <code>LogisticRegression()</code> on them (<code>StratifiedKFold(5, shuffle=True, random_state=0)</code>).</p>
<p>2. <code>honest_cv(X, y)</code>: the same, but with <code>SelectKBest(f_classif, k=20)</code> inside a <code>Pipeline</code> so selection happens within each fold.</p>
<p>Explain the gap. Name two other ways leakage sneaks into a pipeline.</p>

**Theory to know cold**
<ul><li>Every step that learns from data (scaling, imputation, feature selection, target encoding) must be fitted on the training fold only. Pipelines enforce this.</li><li>This is the "wrong way to do cross-validation" from ESL §7.10.2.</li></ul>

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

def leaky_cv(X, y):
    # your code here
    pass

def honest_cv(X, y):
    # your code here
    pass

# Sample run
try:
    print(leaky_cv(noise_X, noise_y), honest_cv(noise_X, noise_y))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("w3")

---
## 4. Walk-forward validation vs shuffled K-fold on a time series
*Hard · id `w4`*

<p><code>walk</code> is a random walk: tomorrow is unpredictable, but neighbouring days are close.</p>
<p>Write <code>cv_r2(series)</code>: features = the time index only (<code>np.arange(n).reshape(-1, 1)</code>), model = <code>RandomForestRegressor(n_estimators=100, random_state=0)</code>. Return <code>(shuffled, walk_forward)</code>: the mean R² from <code>KFold(5, shuffle=True, random_state=0)</code> and from <code>TimeSeriesSplit(5)</code>.</p>
<p>Why does shuffled K-fold look brilliant? What do purging and an embargo add in finance?</p>

**Theory to know cold**
<ul><li>Shuffled folds put a test day between two training days, so the model interpolates: look-ahead bias.</li><li>Walk-forward trains only on the past. With overlapping labels (e.g. 5-day returns), also purge training samples whose labels overlap the test window, and leave an embargo gap.</li></ul>

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score

def cv_r2(series):
    # your code here
    pass

# Sample run
try:
    print(cv_r2(walk))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("w4")